# VQC results on IQM Garnet (Amazon Braket)

This notebook mirrors the IBM VQC analyses (`vqc_ibm_kingston_analysis.ipynb`,
`vqc_ibm_fez_analysis.ipynb`, `vqc_ibm_marrakesh_analysis.ipynb`) and the Braket
Emerald notebook (`vqc_braket_iqm_emerald_analysis.ipynb`): same parity observable,
metrics, plots, ideal merge, and CSV layout.

Raw data are **aggregated histogram JSON** files under `garnet/` at the repo root
(`qvc_test_MMM.json`: `{bitstring: count}`, **1024 shots** per file). The format is
the same histogram JSON convention as Marrakesh / Emerald (flat `qvc_test_*.json` files).

**Subset:** indices **000–010** (`IDX_MIN`…`IDX_MAX` in the next cell). Any missing
`qvc_test_MMM.json` in that range is skipped automatically.

**Ground truth:** section **2a** merges `vqc_ideal_reference_from_qasm.csv` on
`idx` = `test_index` (0–10). Optional `vqc_braket_iqm_garnet_job_labels.csv`
(`job_id`, `true_label`) only fills labels for jobs **without** an ideal row.

*Reads `../garnet/` and CSV helpers in `analysis/`; does not call Braket APIs.*


In [ ]:
import base64
import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-darkgrid")

TARGET_BACKEND = "iqm_garnet"
DATA_SUBDIRS = Path("../data/raw/braket_garnet")
IDX_MIN, IDX_MAX = 0, 10
FLAT_JOB_DIRS = []
N_QUBITS = 10
assert DATA_SUBDIRS.is_dir(), "Folder not found: " + str(DATA_SUBDIRS.resolve())


## 1. Loading utilities

Histogram JSON (`qvc_test_*.json`) is expanded into a per-shot `bits` matrix so all
downstream cells match the IBM notebooks (parity, marginals, entropy, XEB-style
purity, cross-job TV distance, etc.).


In [ ]:
def decode_ibm_sampler_v2(result_path, reg_name=None):
    """Return (bits, counts, probs) for a Sampler V2 result file.

    bits   : ndarray (shots, n_qubits) of {0,1}
    counts : dict bitstring -> shot count
    probs  : dict bitstring -> probability

    The classical-register name varies between jobs (e.g. ``meas`` or ``c``);
    if ``reg_name`` is None we auto-pick the first available register.
    """
    data = json.loads(Path(result_path).read_text())
    results = data["data"][0]["results"]
    if reg_name is None:
        reg_name = next(iter(results))
    blk = results[reg_name]
    shots, n_qubits = blk["shape"]
    raw = base64.b64decode(blk["data"])
    bits = np.unpackbits(
        np.frombuffer(raw, dtype=np.uint8), bitorder="little"
    )[: shots * n_qubits].reshape(shots, n_qubits).astype(int)

    counts = Counter("".join(map(str, row)) for row in bits)
    total = sum(counts.values())
    probs = {b: c / total for b, c in counts.items()}
    return bits, dict(counts), probs


def decode_qvc_counts_json(result_path: Path):
    """Expand histogram {bitstring: count} into per-shot bits (row order arbitrary)."""
    hist = json.loads(Path(result_path).read_text())
    if not hist:
        raise ValueError(f"empty histogram: {result_path}")
    lengths = {len(k) for k in hist}
    if len(lengths) != 1:
        raise ValueError(f"inconsistent bitstring lengths in {result_path}: {lengths}")
    n_q = len(next(iter(hist)))
    total = sum(hist.values())
    if total == 0:
        raise ValueError(f"zero shots in {result_path}")
    bitstrings = list(hist.keys())
    char_matrix = np.array([[int(ch) for ch in bs] for bs in bitstrings], dtype=int)
    counts_arr = np.array([hist[bs] for bs in bitstrings], dtype=int)
    bits = np.repeat(char_matrix, counts_arr, axis=0)
    counts = dict(hist)
    probs = {b: c / total for b, c in counts.items()}
    return bits.astype(int), counts, probs


def load_info(info_path: Path) -> dict:
    info = json.loads(Path(info_path).read_text())
    params = info.get("params", {}) or {}
    return {
        "backend": info.get("backend"),
        "job_id":  info.get("id"),
        "created": info.get("created"),
        "status":  info.get("status"),
        "cost":    info.get("cost"),
        "shots":   params.get("shots"),
        "tags":    info.get("tags"),
    }


def load_job_info(job: dict) -> dict:
    if job.get("histogram_only"):
        return {
            "backend": TARGET_BACKEND,
            "job_id": job["job_id"],
            "created": job.get("created_hint"),
            "status": "COMPLETED",
            "cost": None,
            "shots": None,
            "tags": ("qvc_histogram_json",),
        }
    return load_info(job["info_path"])


def decode_job_measurements(job: dict):
    if job.get("histogram_only"):
        return decode_qvc_counts_json(job["result_path"])
    return decode_ibm_sampler_v2(job["result_path"])


def folder_index(name: str) -> int:
    m = re.search(r"(\d+)$", name)
    return int(m.group(1)) if m else -1


def discover_jobs(root: Path):
    """Return list of dicts: {idx, folder, info_path, result_path}."""
    jobs = []
    for sub in sorted(root.iterdir(), key=lambda p: folder_index(p.name)):
        if not sub.is_dir():
            continue
        info_files = list(sub.glob("*-info.json"))
        result_files = list(sub.glob("*-result.json"))
        if not info_files or not result_files:
            continue
        jobs.append({
            "idx": folder_index(sub.name),
            "folder": sub.name,
            "info_path": info_files[0],
            "result_path": result_files[0],
        })
    return jobs


def discover_flat_job_dirs(flat_roots):
    """Top-level *-info.json + *-result.json pairs (e.g. flat job dumps)."""
    jobs = []
    offset = 0
    for root in flat_roots:
        if not root.is_dir():
            continue
        for info_path in sorted(root.glob("*-info.json")):
            result_path = info_path.with_name(
                info_path.name.replace("-info.json", "-result.json")
            )
            if not result_path.is_file():
                continue
            stem = info_path.name.replace("-info.json", "")
            jobs.append({
                "idx": 1000 + offset,
                "folder": f"{root.name}__{stem}",
                "info_path": info_path,
                "result_path": result_path,
            })
            offset += 1
    return jobs


def discover_garnet_qvc_histograms(root: Path):
    """``qvc_test_MMM.json`` in ``DATA_SUBDIRS``; keep ``IDX_MIN``..``IDX_MAX`` inclusive."""
    jobs = []
    for path in sorted(root.glob("qvc_test_*.json")):
        m = re.match(r"^qvc_test_(\d+)\.json$", path.name)
        if not m:
            continue
        idx = int(m.group(1))
        if idx < IDX_MIN or idx > IDX_MAX:
            continue
        stem = path.stem
        jobs.append({
            "idx": idx,
            "folder": stem,
            "histogram_only": True,
            "job_id": f"iqm_garnet_{stem}",
            "created_hint": f"{stem}.json",
            "info_path": path,
            "result_path": path,
        })
    return jobs


JOBS = discover_garnet_qvc_histograms(DATA_SUBDIRS) + discover_flat_job_dirs(FLAT_JOB_DIRS)
JOBS.sort(key=lambda j: (j["idx"], j["folder"]))
print(
    f"Found {len(JOBS)} IQM Garnet histogram jobs "
    f"(idx {IDX_MIN:03d}..{IDX_MAX:03d}) + flat dirs:"
)
for j in JOBS:
    print(f"  [{j['idx']:04d}] {j['folder']}")


## 2. Decode every job

For each folder we load the metadata, decode the 1024 shots, and compute
$\langle Z^{\otimes 10}\rangle$ together with its sign as the predicted class
label. We also record per-qubit marginals $P(q_i = 1)$ and the Hamming-weight
distribution which is useful to spot bias and depolarising-style noise.


In [ ]:
def expectation_z_all(bits: np.ndarray) -> float:
    """<Z^{\\otimes n}> from raw shot bits."""
    parity = bits.sum(axis=1) % 2  # 0 -> +1, 1 -> -1
    return float((1 - 2 * parity).mean())


records = []
all_data_raw = {}

for j in JOBS:
    info = load_job_info(j)
    bits, counts, probs = decode_job_measurements(j)
    shots, n_q = bits.shape

    ev = expectation_z_all(bits)
    pred = 1 if ev >= 0 else -1
    margin = abs(ev)
    p_q1 = bits.mean(axis=0)
    hw = bits.sum(axis=1)
    hw_counts = Counter(int(x) for x in hw)
    n_unique = len(counts)
    p_arr = np.fromiter(probs.values(), dtype=float)
    entropy = float(-(p_arr * np.log2(p_arr)).sum())
    readout_spread = float(np.max(p_q1) - np.min(p_q1))
    xeb_linear = float((2 ** N_QUBITS) * np.sum(p_arr ** 2))
    parity_even_frac = float((hw % 2 == 0).mean())
    top_bitstring = max(counts.items(), key=lambda kv: kv[1])[0]

    records.append({
        "idx": j["idx"],
        "folder": j["folder"],
        "backend": info["backend"],
        "job_id": info["job_id"],
        "created": info["created"],
        "shots": shots,
        "unique_bitstrings": n_unique,
        "<Z^10>": ev,
        "|<Z^10>|": margin,
        "prediction": pred,
        "H[bits]_bits": entropy,
        "mean_HW": float(hw.mean()),
        "readout_spread": readout_spread,
        "xeb_linear_2n_Ep2": xeb_linear,
        "parity_even_frac": parity_even_frac,
        "top_bitstring": top_bitstring,
    })
    all_data_raw[j["idx"]] = {
        "bits": bits,
        "counts": counts,
        "probs": probs,
        "hw": hw,
        "hw_counts": hw_counts,
        "p_q1": p_q1,
        "backend": info["backend"],
        "job_id": info["job_id"],
        "created": info["created"],
    }

summary_full = pd.DataFrame.from_records(records).sort_values("idx").reset_index(drop=True)
sm = summary_full[summary_full["backend"] == TARGET_BACKEND].copy()
sm = sm.sort_values(["created", "idx"])
sm = sm.drop_duplicates(subset="job_id", keep="first")
summary = sm.sort_values("idx").reset_index(drop=True)

kept_idx = set(summary["idx"])
all_data = {i: all_data_raw[i] for i in kept_idx}

print(
    f"Rows before filter: {len(summary_full)} | "
    f"after {TARGET_BACKEND} + dedupe: {len(summary)}"
)
if len(summary) == 0:
    print("No rows left — add IQM Garnet histogram JSON under DATA_SUBDIRS / FLAT_JOB_DIRS or check paths.")
summary


### Sanity checks on the run set

After restricting to `TARGET_BACKEND` and dropping duplicate `job_id`s, this
section verifies there are no remaining duplicates and lists any non-IQM Garnet
rows that were removed upstream (they appear only in `summary_full` if you
inspect it in the previous cell).


In [ ]:
dup = summary.groupby("job_id").filter(lambda g: len(g) > 1)
if len(dup):
    print("Duplicate job_ids still present (unexpected):")
    print(dup[["idx", "folder", "job_id", "backend"]].to_string(index=False))
else:
    print("No duplicate job_ids in filtered table.")

off = summary_full[summary_full["backend"] != TARGET_BACKEND]
if len(off):
    print(f"\nDropped as backend != {TARGET_BACKEND} ({len(off)} row(s)):")
    print(off[["idx", "folder", "backend", "job_id"]].to_string(index=False))
else:
    print(f"\nNo non-{TARGET_BACKEND} rows in the raw table.")

dup_raw = summary_full.groupby("job_id").filter(lambda g: len(g) > 1)
if len(dup_raw):
    print("\nDuplicate job_ids in raw load (before dedupe):")
    print(dup_raw[["idx", "folder", "job_id", "backend"]].to_string(index=False))


## 2a. Ground truth vs noiseless OpenQASM ideal

`vqc_ideal_reference_from_qasm.csv` (from `ideal_reference_vqc_from_qasm.ipynb`) lists exact \(\langle Z^{\otimes 10}\rangle\), stripe `true_label`, and ideal output statistics **per `test_index`** (`qvc_test_NNN` ↔ `test_index` = N).

We **left-merge** on `summary["idx"] == test_index`. Folder indices **0–10** align with `qvc_test_000`…`010` in the current reference CSV. Rows with `idx` not present in the ideal CSV have no ideal row (`has_ideal_ref` = False).


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    matthews_corrcoef,
)

IDEAL_REF_PATHS = [
    Path("../data/ideal/vqc_ideal_reference_from_qasm.csv"),
    Path("analysis/vqc_ideal_reference_from_qasm.csv"),
]
ideal_csv = next((p for p in IDEAL_REF_PATHS if p.is_file()), None)

if ideal_csv is None:
    print(
        "No vqc_ideal_reference_from_qasm.csv found — run ideal_reference_vqc_from_qasm.ipynb "
        "or place the CSV next to this notebook."
    )
    summary["has_ideal_ref"] = False
else:
    ref = pd.read_csv(ideal_csv)
    ref_cols = [
        "test_index",
        "true_label",
        "ideal_Z_all_expectation",
        "ideal_prediction",
        "ideal_entropy_bits",
        "ideal_linear_xeb_purity",
        "ideal_top_bitstring",
        "ideal_top_prob",
        "ideal_mean_hamming",
    ]
    ref = ref[[c for c in ref_cols if c in ref.columns]]
    summary = summary.merge(ref, left_on="idx", right_on="test_index", how="left")
    summary["has_ideal_ref"] = summary["ideal_Z_all_expectation"].notna()
    missing = summary.loc[~summary["has_ideal_ref"], "idx"].tolist()
    if missing:
        print(
            "No ideal row for some folder idx values (add matching qvc_test_*.qasm or regenerate vqc_ideal_reference_from_qasm.csv):",
            missing,
        )
    summary["Z_err_empirical_minus_ideal"] = summary["<Z^10>"] - summary["ideal_Z_all_expectation"]
    summary["entropy_err_vs_ideal"] = summary["H[bits]_bits"] - summary["ideal_entropy_bits"]
    summary["purity_delta_hw_minus_ideal"] = (
        summary["xeb_linear_2n_Ep2"] - summary["ideal_linear_xeb_purity"]
    )
    summary["true_label"] = pd.to_numeric(summary["true_label"], errors="coerce")
    summary["correct_vs_true_label"] = summary["prediction"] == summary["true_label"]
    summary["correct_vs_ideal_prediction"] = summary["prediction"] == summary["ideal_prediction"]

    sub = summary[summary["has_ideal_ref"]]
    if len(sub):
        yt = sub["true_label"].astype(int).to_numpy()
        yp = sub["prediction"].astype(int).to_numpy()
        print(
            f"\nGround-truth classification (n={len(sub)} jobs with OpenQASM ideal row):"
        )
        print(f"  accuracy vs stripe label:     {accuracy_score(yt, yp):.4f}")
        print(f"  balanced accuracy:            {balanced_accuracy_score(yt, yp):.4f}")
        print(f"  Cohen kappa:                  {cohen_kappa_score(yt, yp):.4f}")
        print(f"  Matthews correlation:         {matthews_corrcoef(yt, yp):.4f}")
        cm = confusion_matrix(yt, yp, labels=[-1, 1])
        fig, ax = plt.subplots(figsize=(4.0, 3.4))
        im = ax.imshow(cm, cmap="Greens")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["pred −1", "pred +1"])
        ax.set_yticklabels(["true −1", "true +1"])
        for (i, j), v in np.ndenumerate(cm):
            ax.text(
                j,
                i,
                str(v),
                ha="center",
                va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
                fontsize=11,
            )
        ax.set_title("Confusion (hardware vs stripe label, ideal-ref rows)")
        plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        plt.show()

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
    s = summary[summary["has_ideal_ref"]]
    if len(s):
        ax = axes[0]
        x = s["ideal_Z_all_expectation"].to_numpy()
        y = s["<Z^10>"].to_numpy()
        lim = min(-1.05, x.min(), y.min()), max(1.05, x.max(), y.max())
        ax.scatter(x, y, c=s["idx"], cmap="tab10", s=70, edgecolors="k", linewidths=0.4)
        for _, r in s.iterrows():
            ax.annotate(
                str(int(r["idx"])),
                (r["ideal_Z_all_expectation"], r["<Z^10>"]),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=8,
            )
        ax.plot(lim, lim, "k--", lw=1, label="y = x")
        ax.set_xlim(lim)
        ax.set_ylim(lim)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlabel(r"ideal $\langle Z^{10}\rangle$ (noiseless QASM)")
        ax.set_ylabel(r"hardware $\langle Z^{10}\rangle$ (1024 shots)")
        ax.set_title("Parity expectation: hardware vs ideal")
        ax.legend(loc="lower right", fontsize=8)

        ax = axes[1]
        ax.bar(
            s["idx"].astype(str),
            s["Z_err_empirical_minus_ideal"],
            color=np.where(s["Z_err_empirical_minus_ideal"] >= 0, "#3182bd", "#de2d26"),
            edgecolor="k",
            linewidth=0.35,
        )
        ax.axhline(0, color="k", lw=0.9)
        ax.set_xlabel("folder idx")
        ax.set_ylabel(r"$\Delta\langle Z^{10}\rangle$ (hw − ideal)")
        ax.set_title("Bias vs ideal parity channel")

        ax = axes[2]
        ax.bar(s["idx"].astype(str), s["entropy_err_vs_ideal"], color="#756bb1", edgecolor="k", linewidth=0.35)
        ax.axhline(0, color="k", lw=0.9)
        ax.set_xlabel("folder idx")
        ax.set_ylabel("Δ entropy (bits, hw − ideal)")
        ax.set_title("Output distribution entropy vs ideal")
    else:
        for ax in axes:
            ax.text(0.5, 0.5, "no ideal rows", ha="center", va="center", transform=ax.transAxes)
    plt.tight_layout()
    plt.show()

summary


## 2b. Metrics: accuracy-related scores, CIs, readout spread, XEB-style purity, parity

Section **2a** attaches `true_label` and ideal \(\langle Z^{10}\rangle\) from `vqc_ideal_reference_from_qasm.csv` where `idx` matches `test_index` (0–10, matching the current `qvc_test_*.qasm` set).

Here we add **optional** `vqc_braket_iqm_garnet_job_labels.csv` (`job_id`, `true_label`) only to fill labels for rows **without** an ideal row (if you ever have additional jobs beyond the reference CSV). When every row has `true_label` from 2a + CSV, we report **balanced accuracy**, **Cohen’s $\kappa$**, **Matthews correlation**, **confusion matrix**, **bootstrap** mean accuracy with percentile CI, and a **Wilson binomial CI** vs random \(0.5\). If some rows still lack labels, supervised lines are skipped for the full table but 2a already showed metrics for the ideal-matched subset.

We also report **readout spread** (range of single-qubit $P(q_i{=}1)$), **linear XEB-style purity** $2^n\sum_x p(x)^2$ (equals $1$ for uniform noise), the **fraction of shots with even global parity**, and bootstrap uncertainty for the mean **confidence margin** $|\langle Z^{\otimes 10}\rangle|$.


In [ ]:
from scipy import stats
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    matthews_corrcoef,
)

LABELS_CSV = Path("vqc_braket_iqm_garnet_job_labels.csv")

n_jobs = len(summary)
y_pred = summary["prediction"].to_numpy() if n_jobs else np.array([])

if n_jobs == 0:
    print("No jobs in filtered summary — skip metrics block.")
else:
    merged = summary.copy()
    if "true_label" in merged.columns:
        merged["true_label"] = pd.to_numeric(merged["true_label"], errors="coerce")
    else:
        merged["true_label"] = np.nan
    if LABELS_CSV.is_file():
        lab = pd.read_csv(LABELS_CSV)
        lab.columns = lab.columns.str.strip()
        if "true_label" not in lab.columns or "job_id" not in lab.columns:
            print(
                f"{LABELS_CSV}: need columns job_id, true_label "
                f"(found: {list(lab.columns)})"
            )
        else:
            extra = merged.merge(
                lab[["job_id", "true_label"]].rename(columns={"true_label": "true_label_csv"}),
                on="job_id",
                how="left",
            )
            tl_csv = pd.to_numeric(extra["true_label_csv"], errors="coerce")
            merged["true_label"] = merged["true_label"].fillna(tl_csv)
    merged["true_label"] = pd.to_numeric(merged["true_label"], errors="coerce")
    labeled = merged["true_label"].notna()
    if labeled.all() and labeled.any():
        y_true = merged["true_label"].astype(int).to_numpy()
        acc = accuracy_score(y_true, y_pred)
        bacc = balanced_accuracy_score(y_true, y_pred)
        kap = cohen_kappa_score(y_true, y_pred)
        mcc = matthews_corrcoef(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred, labels=[-1, 1])
        print(f"Accuracy:              {acc:.4f}")
        print(f"Balanced accuracy:     {bacc:.4f}")
        print(f"Cohen kappa:           {kap:.4f}")
        print(f"Matthews correlation:  {mcc:.4f}")
        rng = np.random.default_rng(42)
        B = 5000
        acc_boot = []
        for _ in range(B):
            idx = rng.integers(0, n_jobs, size=n_jobs)
            acc_boot.append(accuracy_score(y_true[idx], y_pred[idx]))
        acc_boot = np.array(acc_boot)
        print(
            "Bootstrap mean accuracy: "
            f"{acc_boot.mean():.4f}, 95% CI [{np.percentile(acc_boot, 2.5):.4f}, "
            f"{np.percentile(acc_boot, 97.5):.4f}]"
        )
        k_ok = int((y_true == y_pred).sum())
        bt = stats.binomtest(k_ok, n_jobs, p=0.5)
        ci = bt.proportion_ci(confidence_level=0.95, method="wilson")
        print(
            f"Binomial vs p=0.5: accuracy={k_ok / n_jobs:.4f}, "
            f"Wilson 95% CI [{ci.low:.4f}, {ci.high:.4f}]"
        )
        fig, ax = plt.subplots(figsize=(4.2, 3.6))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(["pred −1", "pred +1"])
        ax.set_yticklabels(["true −1", "true +1"])
        for (i, j), v in np.ndenumerate(cm):
            ax.text(
                j, i, str(v), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black",
            )
        ax.set_title("Confusion matrix (labeled jobs)")
        plt.colorbar(im, ax=ax, fraction=0.046)
        plt.tight_layout()
        plt.show()
    else:
        print(
            f"Supervised metrics skipped — add {LABELS_CSV} with job_id,true_label "
            "for every row in `summary`, or leave partial labels out."
        )

    k_pos = int((y_pred == 1).sum())
    btp = stats.binomtest(k_pos, n_jobs, p=0.5)
    cip = btp.proportion_ci(confidence_level=0.95, method="wilson")
    print(
        f"P(pred = +1) = {k_pos / n_jobs:.4f}, vs 0.5 Wilson 95% CI "
        f"[{cip.low:.4f}, {cip.high:.4f}]"
    )

    margin = summary["|<Z^10>|"].to_numpy()
    rng2 = np.random.default_rng(1)
    m_boot = np.array([
        margin[rng2.integers(0, n_jobs, size=n_jobs)].mean() for _ in range(5000)
    ])
    print(
        "Bootstrap mean |<Z^10>|: "
        f"{m_boot.mean():.4f}, 95% CI [{np.percentile(m_boot, 2.5):.4f}, "
        f"{np.percentile(m_boot, 97.5):.4f}]"
    )

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
    x = np.arange(n_jobs)
    axes[0].bar(x, summary["readout_spread"], color="steelblue", edgecolor="black", linewidth=0.3)
    axes[0].set_xticks(x)
    jid = summary["job_id"].astype(str).str[-8:]
    axes[0].set_xticklabels(jid, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel(r"max$_i P(q_i{=}1)$ − min$_i$")
    axes[0].set_title("Readout spread (per-qubit marginals)")

    axes[1].bar(x, summary["xeb_linear_2n_Ep2"], color="darkorange", edgecolor="black", linewidth=0.3)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(jid, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylabel(r"$2^n \sum_x p(x)^2$")
    axes[1].set_title("Linear XEB-style purity (1 = uniform)")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 3.6))
    cols = np.where(summary["prediction"].to_numpy() > 0, "#1b9e77", "#d95f02")
    ax.bar(summary["idx"], summary["parity_even_frac"], color=cols, edgecolor="black", linewidth=0.3)
    ax.axhline(0.5, color="k", linestyle="--", linewidth=0.9)
    ax.set_xlabel("job idx (folder ordering)")
    ax.set_ylabel("fraction even-parity shots")
    ax.set_title("Global output parity (fraction of shots with |x| even)")
    plt.tight_layout()
    plt.show()


## 3. Predictions per job

Each VQC inference outputs $\langle Z^{\otimes 10}\rangle\in[-1,+1]$, and the
predicted class is $\mathrm{sign}(\langle Z^{\otimes 10}\rangle)$. The bar chart
below colours each job by its predicted class and annotates the magnitude of
the expectation value, which doubles as the model's *confidence margin*.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
x = summary["idx"].to_numpy()
y = summary["<Z^10>"].to_numpy()
colors = ["#2c7fb8" if v >= 0 else "#d7191c" for v in y]
bars = ax.bar(x, y, color=colors, edgecolor="black")
ax.axhline(0, color="black", lw=0.8)
for b, v in zip(bars, y):
    ax.text(b.get_x() + b.get_width() / 2, v + (0.01 if v >= 0 else -0.04),
            f"{v:+.3f}", ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f"{i:02d}" for i in x])
ax.set_xlabel("Job index")
ax.set_ylabel(r"$\langle Z^{\otimes 10}\rangle$")
ax.set_title("VQC inference output per IQM Garnet (Braket) job\n"
             "blue bars => predicted +1 (vertical line), red bars => -1 (horizontal line)")
ax.set_ylim(-1.05, 1.05)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(summary["idx"], summary["|<Z^10>|"], color="#5e3c99", edgecolor="black")
ax.set_xticks(summary["idx"])
ax.set_xticklabels([f"{i:02d}" for i in summary["idx"]])
ax.set_xlabel("Job index")
ax.set_ylabel(r"$|\langle Z^{\otimes 10}\rangle|$")
ax.set_title("Confidence margin |<Z^10>| per job (closer to 1 = more decisive)")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print(f"Mean |<Z^10>| over jobs: {summary['|<Z^10>|'].mean():.3f}")
print(f"Mean Shannon entropy of bitstring dist: {summary['H[bits]_bits'].mean():.2f} / {N_QUBITS} bits max")


## 4. Hamming-weight distributions

On a noiseless circuit, the parity $Z^{\otimes 10}$ depends only on the parity
of the Hamming weight (`HW % 2`). Plotting the HW histogram makes the global
parity easy to read off, and any drift towards the centre (HW $\approx 5$)
indicates depolarising-style noise pulling the distribution to the maximally
mixed state.


In [ ]:
n_jobs = len(all_data)
ncols = 4
nrows = int(np.ceil(n_jobs / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 2.6 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

weights_axis = np.arange(N_QUBITS + 1)
for ax, (idx, d) in zip(axes, sorted(all_data.items())):
    counts = np.array([d["hw_counts"].get(w, 0) for w in weights_axis])
    parity = weights_axis % 2
    colors = ["#2c7fb8" if p == 0 else "#d7191c" for p in parity]
    ax.bar(weights_axis, counts, color=colors, edgecolor="black", linewidth=0.4)
    ev = expectation_z_all(d["bits"])
    ax.set_title(f"job {idx:02d}  ({d['backend']})\n<Z^10>={ev:+.3f}", fontsize=9)
    ax.set_xticks(weights_axis)

for ax in axes[n_jobs:]:
    ax.axis("off")

fig.supxlabel("Hamming weight |x|")
fig.supylabel("Shot count")
fig.suptitle("Hamming-weight distributions "
             "(blue bars: even parity / +1 contribution, red: odd / -1)")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 5. Per-qubit marginals $P(q_i = 1)$

These are useful for spotting biased / hot qubits. On IQM Garnet the readout
asymmetry is non-negligible, so we expect deviations from `0.5` even for
highly-mixed circuits.


In [ ]:
marg = np.stack([all_data[i]["p_q1"] for i in sorted(all_data)], axis=0)
labels = [f"job {i:02d}" for i in sorted(all_data)]

fig, ax = plt.subplots(figsize=(9, 0.5 * len(labels) + 1.5))
im = ax.imshow(marg, cmap="viridis", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(N_QUBITS))
ax.set_xticklabels([f"q{i}" for i in range(N_QUBITS)])
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels)
for i in range(marg.shape[0]):
    for j in range(marg.shape[1]):
        ax.text(j, i, f"{marg[i, j]:.2f}", ha="center", va="center",
                color="white" if marg[i, j] < 0.55 else "black", fontsize=8)
fig.colorbar(im, ax=ax, label="P(q_i = 1)")
ax.set_title("Per-qubit marginal P(q_i = 1) across jobs")
plt.tight_layout()
plt.show()

print("Mean P(q=1) across jobs and qubits :", f"{marg.mean():.3f}")
print("Std of mean per-qubit P(q=1)        :", f"{marg.mean(axis=0).std():.3f}")


## 6. Top bitstrings per job

What does the output distribution look like? For each job we list the most
frequent bitstrings and the cumulative probability they cover.


In [ ]:
TOP_K = 8
rows = []
for idx in sorted(all_data):
    d = all_data[idx]
    items = sorted(d["counts"].items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    total = sum(d["counts"].values())
    cum = 0
    for rank, (bs, c) in enumerate(items, start=1):
        cum += c
        rows.append({
            "job": idx,
            "rank": rank,
            "bitstring": bs,
            "count": c,
            "prob": c / total,
            "cum_prob_topK": cum / total,
            "parity": bs.count("1") % 2,
        })
top_df = pd.DataFrame(rows)
top_df


In [ ]:
fig, axes = plt.subplots(int(np.ceil(len(all_data) / 4)), 4,
                         figsize=(14, 2.4 * int(np.ceil(len(all_data) / 4))),
                         sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, idx in zip(axes, sorted(all_data)):
    d = all_data[idx]
    items = sorted(d["counts"].items(), key=lambda kv: kv[1], reverse=True)[:TOP_K]
    bs = [b for b, _ in items]
    pr = [c / sum(d["counts"].values()) for _, c in items]
    pc = ["#2c7fb8" if b.count("1") % 2 == 0 else "#d7191c" for b in bs]
    ax.bar(range(len(items)), pr, color=pc, edgecolor="black", linewidth=0.4)
    ax.set_xticks(range(len(items)))
    ax.set_xticklabels(bs, rotation=90, fontsize=7, family="monospace")
    ax.set_title(f"job {idx:02d}", fontsize=9)
    ax.set_ylim(0, max(0.05, max(pr) * 1.2))
for ax in axes[len(all_data):]:
    ax.axis("off")
fig.supylabel("probability")
fig.suptitle(f"Top-{TOP_K} bitstrings per job (blue = even parity, red = odd parity)")
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 7. Cross-job similarity

Are the IQM Garnet runs sampling from notably different distributions, or are some
near-duplicates? We compute the total-variation distance
$TV(p,q) = \tfrac{1}{2}\sum_x |p(x) - q(x)|$ over all observed bitstrings.


In [ ]:
def tv_distance(p, q):
    keys = set(p) | set(q)
    return 0.5 * sum(abs(p.get(k, 0.0) - q.get(k, 0.0)) for k in keys)

idxs = sorted(all_data)
M = np.zeros((len(idxs), len(idxs)))
for i, a in enumerate(idxs):
    for j, b in enumerate(idxs):
        if i <= j:
            M[i, j] = tv_distance(all_data[a]["probs"], all_data[b]["probs"])
            M[j, i] = M[i, j]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(M, cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(len(idxs)))
ax.set_xticklabels([f"{i:02d}" for i in idxs])
ax.set_yticks(range(len(idxs)))
ax.set_yticklabels([f"{i:02d}" for i in idxs])
for i in range(len(idxs)):
    for j in range(len(idxs)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                fontsize=7, color="white" if M[i, j] < 0.5 else "black")
fig.colorbar(im, ax=ax, label="TV distance")
ax.set_title("Total-variation distance between job bitstring distributions")
plt.tight_layout()
plt.show()


## 8. Final summary table

A clean view of the per-job metrics. Save it to CSV for later inclusion in the
paper.


In [ ]:
out_csv = Path("../data/processed/vqc_braket_iqm_garnet_summary.csv")
summary.to_csv(out_csv, index=False)
print(f"Saved {out_csv.resolve()}")
summary


## 9. Take-aways

* Each IBM run delivers $\langle Z^{\otimes 10}\rangle$ values whose magnitude is
  generally well below 1 (Hamming-weight histograms are wide and centred near
  $\sim 5$). This is consistent with the expected depolarising-style noise on
  the 10-qubit transpiled circuit.
* The sign of $\langle Z^{\otimes 10}\rangle$ is the actual VQC class
  prediction; the bar chart in section 3 shows which inputs were classified as
  vertical (`+1`) vs horizontal (`-1`).
* Per-qubit marginals reveal which physical qubits picked up readout bias on
  the day of the run; this is helpful when comparing against the Z-feature
  encoding which is supposed to be near-balanced for our 2x5 image inputs.
* The TV-distance heatmap compares **unique** IQM Garnet jobs (duplicates are
  removed before plotting).
* The summary table is exported to `vqc_braket_iqm_garnet_summary.csv` (includes ideal
  columns when `vqc_ideal_reference_from_qasm.csv` is present). Section **2a**
  compares hardware to noiseless OpenQASM; optional `vqc_braket_iqm_garnet_job_labels.csv`
  fills labels only for folder indices not covered by the ideal CSV.
